In [3]:
import pandas as pd
from sklearn .preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [4]:
df = pd.read_csv("dataset_spending_19.csv")
print(df.head())
print(df.info())


   CustomerID  Age  Income_$  SpendingScore  VisitsPerMonth  OnlinePurchases  \
0           1   28        33             78              14                9   
1           2   21        25             87               8               23   
2           3   23        24             88              13               10   
3           4   24        25             73              16               11   
4           5   20        23             88              17               16   

   Gender Region  
0  Female   East  
1    Male  North  
2    Male  South  
3  Female   West  
4    Male   West  
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CustomerID       200 non-null    int64
 1   Age              200 non-null    int64
 2   Income_$         200 non-null    int64
 3   SpendingScore    200 non-null    int64
 4   VisitsPerMonth   200 non-null    int64
 5  

In [5]:
FEATURES = [ 'Income_$', "SpendingScore"]
X = df[FEATURES].copy()
print(X.head()) 

   Income_$  SpendingScore
0        33             78
1        25             87
2        24             88
3        25             73
4        23             88


In [6]:
for col in FEATURES:
 if X[col].isna().any():
  X[col] = X[col].fillna(X[col].median())


# scale

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled[:5])
  


[[-0.62783049  0.72998073]
 [-0.89031514  1.07359091]
 [-0.92312573  1.11176982]
 [-0.89031514  0.53908619]
 [-0.95593631  1.11176982]]


In [7]:
print("elboW method:")
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    print(f"k={k}  -> SSE={km.inertia_:.2f}")

elboW method:
k=1  -> SSE=400.00
k=2  -> SSE=199.70
k=3  -> SSE=79.37
k=4  -> SSE=21.37
k=5  -> SSE=19.09
k=6  -> SSE=15.65
k=7  -> SSE=14.48
k=8  -> SSE=13.81
k=9  -> SSE=12.94
k=10  -> SSE=11.52


In [8]:
K = 6

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = labels.astype(int)

print("sample of clustered data:")
print(df.head())

sample of clustered data:
   CustomerID  Age  Income_$  SpendingScore  VisitsPerMonth  OnlinePurchases  \
0           1   28        33             78              14                9   
1           2   21        25             87               8               23   
2           3   23        24             88              13               10   
3           4   24        25             73              16               11   
4           5   20        23             88              17               16   

   Gender Region  Cluster  
0  Female   East        2  
1    Male  North        4  
2    Male  South        4  
3  Female   West        2  
4    Male   West        4  


In [9]:
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)

print(f"Silhouette Score: {sil:.4f}")
print(f"Davies-Bouldin Index: {dbi:.4f}")

Silhouette Score: 0.5713
Davies-Bouldin Index: 0.7519


In [10]:

centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)


center_df = pd.DataFrame(centers_original, columns=FEATURES)

print("changed to original unit :" , center_df.round(2))

changed to original unit :    Income_$  SpendingScore
0     59.90          58.55
1     28.92          19.60
2     25.33          78.04
3     99.16          79.24
4     22.74          89.04
5     51.38          46.71


In [11]:
sample_idx = [0,2,3]
sanity_check = df.loc[sample_idx, FEATURES + ["Cluster"]]
print("Sanity check for original (3 customers):")
print(sanity_check)

Sanity check for original (3 customers):
   Income_$  SpendingScore  Cluster
0        33             78        2
2        24             88        4
3        25             73        2


In [16]:
OUTH_PATH = "spending_labelded_cluster.csv"
df.to_csv(OUTH_PATH , index=False)
